In [ ]:
import pandas as pd
import numpy as np
from getpass import getuser

user = getuser()
base = rf'C:\Users\{user}\Documents\GitHub\tb_football'

# Elo snapshot files: one rating per team per tournament, used to fill teams
# whose matches have no goals (so they never appear in elo_home/elo_away columns)
elo_supp_wc = pd.read_excel(rf'{base}\data\in\elo_wc.xlsx').rename(columns={'elo_rating': 'elo'})
elo_supp_eu = pd.read_excel(rf'{base}\data\in\elo_eu.xlsx').rename(columns={'elo_rating': 'elo'})

## 1. Merge goals datasets

In [ ]:
goals_wc = pd.read_excel(rf'{base}\data\out\wiki\men\fifa\wc\goals_wc_fifa.xlsx')
goals_eu = pd.read_excel(rf'{base}\data\out\wiki\men\uefa\eu\goals_eu_uefa.xlsx')

goals_wc['fifa_rule'] = 1
goals_eu['fifa_rule'] = 0

goals = pd.concat([goals_wc, goals_eu], ignore_index=True)
print(f'goals_wc: {len(goals_wc)} rows, goals_eu: {len(goals_eu)} rows, merged: {len(goals)} rows')

### 1a. elo_favorite and elo_underdog

In [ ]:
goals['elo_favorite'] = goals[['elo_home', 'elo_away']].max(axis=1)
goals['elo_underdog'] = goals[['elo_home', 'elo_away']].min(axis=1)

print(goals[['elo_home', 'elo_away', 'elo_favorite', 'elo_underdog']].head())

### 1b. elo1st, elo2nd, elo3rd, elo4th

For each group (year, stage), collect all four teams' elo values and rank them from highest to lowest.

In [ ]:
def add_group_elo_ranks_v2(df, elo_supp_wc, elo_supp_eu):
    """Rank teams by elo within each (year, stage) group, assign elo1st-elo4th.

    Primary source: elo_home / elo_away columns on goal rows.
    Fallback: elo snapshot files for teams that appear in the 1st-4th standings
    columns but have no goal rows (e.g. teams that played only 0-0 matches).
    """
    home = df[['year', 'stage', 'home_team', 'elo_home']].dropna(subset=['home_team', 'elo_home'])
    home = home.rename(columns={'home_team': 'team', 'elo_home': 'elo'})

    away = df[['year', 'stage', 'away_team', 'elo_away']].dropna(subset=['away_team', 'elo_away'])
    away = away.rename(columns={'away_team': 'team', 'elo_away': 'elo'})

    team_elo = (pd.concat([home, away])
                .drop_duplicates(subset=['year', 'stage', 'team']))

    # Supplement: for each group, check standings columns and fill missing teams
    # from the tournament-level elo snapshot files
    grp_meta = (df[['year', 'stage', 'fifa_rule', '1st', '2nd', '3rd', '4th']]
                .drop_duplicates(['year', 'stage']))
    supplement_rows = []
    for _, row in grp_meta.iterrows():
        yr, st, fifa = row['year'], row['stage'], row['fifa_rule']
        supp = elo_supp_wc if fifa == 1 else elo_supp_eu
        yr_supp = supp[supp['year'] == yr]
        for col in ['1st', '2nd', '3rd', '4th']:
            team = row[col]
            if pd.isna(team):
                continue
            already = ((team_elo['year'] == yr) & (team_elo['stage'] == st) & (team_elo['team'] == team)).any()
            if not already:
                match = yr_supp[yr_supp['team'] == team]
                if len(match) > 0:
                    supplement_rows.append({'year': yr, 'stage': st, 'team': team, 'elo': match['elo'].iloc[0]})
                else:
                    print(f'  WARNING: no elo found for {team} ({yr})')

    if supplement_rows:
        team_elo = (pd.concat([team_elo, pd.DataFrame(supplement_rows)], ignore_index=True)
                    .drop_duplicates(subset=['year', 'stage', 'team']))
        print(f'Supplemented {len(supplement_rows)} missing team elos from snapshot files')

    team_elo = team_elo.sort_values(['year', 'stage', 'elo'], ascending=[True, True, False])

    records = []
    for (yr, st), grp in team_elo.groupby(['year', 'stage']):
        elos = grp['elo'].values
        rec = {'year': yr, 'stage': st}
        for i, name in enumerate(['elo1st', 'elo2nd', 'elo3rd', 'elo4th']):
            rec[name] = elos[i] if i < len(elos) else np.nan
        records.append(rec)

    group_elos = pd.DataFrame(records)
    return df.merge(group_elos, on=['year', 'stage'], how='left')


goals = add_group_elo_ranks_v2(goals, elo_supp_wc, elo_supp_eu)
print(goals[['year', 'stage', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']].drop_duplicates(['year', 'stage']).head(10))

### 1c. Save merged goals dataset

In [ ]:
out_path = rf'{base}\data\out\goals_merged.xlsx'
goals.to_excel(out_path, index=False)
print(f'Saved goals_merged.xlsx: {len(goals)} rows, {len(goals.columns)} columns')
print('Columns:', list(goals.columns))

## 2. Merge mbm datasets

In [ ]:
mbm_wc = pd.read_excel(rf'{base}\data\out\wiki\men\fifa\wc\mbm_wc_fifa.xlsx')
mbm_eu = pd.read_excel(rf'{base}\data\out\wiki\men\uefa\eu\mbm_eu_uefa.xlsx')

mbm_wc['fifa_rule'] = 1
mbm_eu['fifa_rule'] = 0

mbm = pd.concat([mbm_wc, mbm_eu], ignore_index=True)
print(f'mbm_wc: {len(mbm_wc)} rows, mbm_eu: {len(mbm_eu)} rows, merged: {len(mbm)} rows')

In [ ]:
mbm = add_group_elo_ranks_v2(mbm, elo_supp_wc, elo_supp_eu)
print(mbm[['year', 'stage', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']].drop_duplicates(['year', 'stage']).head(10))

In [ ]:
out_path_mbm = rf'{base}\data\out\mbm_merged.xlsx'
mbm.to_excel(out_path_mbm, index=False)
print(f'Saved mbm_merged.xlsx: {len(mbm)} rows, {len(mbm.columns)} columns')

## 3. Quick check

In [ ]:
print('=== goals_merged ===')
print(goals.groupby('fifa_rule')[['qual_changed', 'elo_favorite', 'elo_underdog',
                                   'elo1st', 'elo2nd', 'elo3rd', 'elo4th']].mean().round(1))

print('\n=== mbm_merged ===')
print(mbm.groupby('fifa_rule')[['suspense', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']].mean().round(1))